# MEDUSA: SFT & GRPO Training Pipeline

This notebook contains the complete workflow to fine-tune a causal language model (e.g., `Qwen2.5-3B` or `1.5B`) to act as an Autonomous Data Engineer inside the MEDUSA environment.

**Pipeline Architecture:**
1. Generate Expert synthetic trajectories (`generate_sft_dataset.py`).
2. Learn basic syntax and environment operations via *Supervised Fine-Tuning* (SFT).
3. Align generalizability via *Generative Reward Proximal Policy Optimization* (GRPO) communicating with the `.openenv` Hugging Face Space.

In [ ]:
# Install dependencies
!pip install --quiet trl transformers datasets openenv_client peft

## 1. Fast SFT Synthetic Dataset Generation
We dynamically generate hundreds of random data engineering pipelines to prevent the LLM from overfitting to specific schema names. Our rule-based expert solver will perfectly navigate these pipelines to produce Ground Truth SFT data.

In [ ]:
!python3 scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl

## 2. Dataset Loading & Smoke Testing
Validating that the dataset was accurately generated and that the columns are properly structured for `trl`.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="medusa_sft_100_episodes.jsonl")

print(f"Loaded {len(ds['train'])} SFT interaction steps.")

# SMOKE TEST 1: Size Validation
assert len(ds['train']) > 1000, "Error: Too few steps produced. Check generation script!"

# Preview format
print("Sample ChatML Trace:")
print(ds['train'][0]['messages'])

print("\n[SMOKE TEST 1 PASSED] SFT Synthetic generation verified.")

## 3. Supervised Fine Tuning (SFT)
Using `trl`'s `SFTTrainer` with LoRA to adapt the model quickly and with low VRAM footprint.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure SFT Hyperparams. SFTConfig automatically supports ChatML "messages" format.
sft_config = SFTConfig(
    output_dir="./medusa-qwen-sft",
    max_seq_length=2048,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=2,
    logging_steps=10,
)

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"]
)

print("SFT Configuration Ready. Initialize `AutoModelForCausalLM` and `SFTTrainer` to train!")

# =========================================================================
# UNCOMMENT TO RUN TRAINING
# =========================================================================
# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
# trainer = SFTTrainer(
#    model=model,
#    train_dataset=ds['train'],
#    peft_config=peft_config,
#    args=sft_config,
# )    
# trainer.train()
# trainer.save_model("medusa-qwen-sft-final")

## 4. GRPO (Generative Reward PPO) - RL Stage
Once SFT provides basic JSON syntax conformity, we connect to `anubhavkamal/medusa-env` via `OpenEnv` to train the model to *think* and execute pipelines autonomously.

In [ ]:
from openenv_client import EnvironmentClient

def remote_smoke_test():
    try:
        env = EnvironmentClient("anubhavkamal/medusa-env")
        obs, info = env.reset()
        
        # Print confirmation it works
        _keys = list(obs.keys())
        print(f"Success! Connected to Medusa HF Space. Observation keys: {_keys}")
        return True
    except Exception as e:
        print(f"Smoke Test Failed! Hugging Face Space might be asleep or unavailable: {e}")
        return False

# SMOKE TEST 2: Verify environment is online
assert remote_smoke_test(), "Space must be awake for GRPO."
print("\n[SMOKE TEST 2 PASSED] Hugging Face Deployment Verified.")

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import json
import re

grpo_config = GRPOConfig(
    output_dir="./medusa-qwen-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_prompt_length=1500,
    max_completion_length=150,
)

def medusa_reward_function(completions, prompts):
    """
    Concept Reward Function for GRPO:
    Uses openenv to step() through Medusa using the model's textual responses.
    Reward flows directly back to the TRPO/PPO policy!
    """
    rewards = []
    env = EnvironmentClient("anubhavkamal/medusa-env")
    
    for completion in completions:
        # Parse JSON identically to env.parse_llm_action()
        clean_text = completion.strip()
        match = re.search(r"```(?:json)?\s*(.*?)\s*```", clean_text, re.DOTALL)
        if match:
            clean_text = match.group(1).strip()
            
        try:
            payload = json.loads(clean_text)
            action = {"action": payload.get("action", "DO_NOTHING"), "params": payload.get("params", {})}
            
            # Send to HF environment and get actual RL reward back!
            obs, reward, terminated, truncated, info = env.step(action)
            rewards.append(float(reward))
            
        except Exception:
            # Formatting penalty for malformed JSON
            rewards.append(-1.0)

    return rewards

print("GRPO Pipeline configured!")